# 08.2 WaveNet：因果卷积、扩张卷积与感受野

本 Notebook 关注 WaveNet 的核心结构：为什么自回归波形生成不能看未来、扩张卷积如何扩大感受野、mu-law token 如何把连续波形变成分类问题。


In [ ]:
from pathlib import Path
import sys

# 路径推断：从 cwd 向上找含 CODE/chapter08/_common 的目录；ROOT 指向 CODE/chapter08/
_p = Path.cwd()
while not (_p / "CODE" / "chapter08" / "_common").exists():
    _parent = _p.parent
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/chapter08/_common 的目录），请在项目内运行本 Notebook")
    _p = _parent
ROOT = _p / "CODE" / "chapter08"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Audio, display

from _common.audio_io import mu_law_decode, mu_law_encode, save_audio
from _common.plotting import finish_figure, plot_spectrogram, setup_plot_style
from synthesis.waveforms import harmonic_stack
from wavenet.model import WaveNet
from wavenet.receptive_field import (
    dependency_matrix,
    dilation_cycle,
    plot_receptive_field,
    receptive_field_size,
)

OUTPUT_FIGURES = ROOT / "output_figures"
OUTPUT_AUDIO = ROOT / "output_audio" / "08_2"
OUTPUT_FIGURES.mkdir(parents=True, exist_ok=True)
OUTPUT_AUDIO.mkdir(parents=True, exist_ok=True)
setup_plot_style()
torch.manual_seed(0)


In [ ]:
dilations = dilation_cycle(layers_per_cycle=6, cycles=2)
rf = receptive_field_size(kernel_size=2, dilations=dilations)
table = pd.DataFrame(
    {
        "layer": np.arange(1, len(dilations) + 1),
        "dilation": dilations,
        "receptive_field_after_layer": [
            receptive_field_size(2, dilations[:i]) for i in range(1, len(dilations) + 1)
        ],
    }
)
print(f"Final receptive field: {rf} samples")
display(table)


In [ ]:
fig = plot_receptive_field(
    sequence_length=96,
    kernel_size=2,
    dilations=dilations,
    out_path=OUTPUT_FIGURES / "08_2_wavenet_receptive_field.png",
)
plt.show()


**因果依赖矩阵**

三层因果卷积（膨胀因子 1、2、4）中，每个输出采样点只依赖自身与更早的输入，黑色标出存在依赖的位置。未来输入一侧全白，单元末尾的断言以输出 4 对输入 5 为例验证了这一点。


In [ ]:
short_dilations = (1, 2, 4)
deps = dependency_matrix(sequence_length=16, kernel_size=2, dilations=short_dilations)

fig, ax = plt.subplots(figsize=(6, 4))
ax.imshow(deps, origin="lower", aspect="auto", cmap="gray_r")
ax.set_xlabel("输入采样点")
ax.set_ylabel("输出采样点")
finish_figure(fig, OUTPUT_FIGURES / "08_2_causal_dependency_matrix.png")
plt.show()

assert not deps[4, 5], "causal convolution must not depend on future input"


In [ ]:
sr = 16000
audio = harmonic_stack(fundamental=220, duration=1.0, sr=sr, harmonics=8)
tokens = mu_law_encode(audio, quantization_channels=256)
decoded = mu_law_decode(tokens, quantization_channels=256)
save_audio(OUTPUT_AUDIO / "harmonic_original.wav", audio, sr)
save_audio(OUTPUT_AUDIO / "harmonic_mulaw_decoded.wav", decoded, sr)

fig, axes = plt.subplots(2, 1, figsize=(9, 4), sharex=True)
t = np.arange(audio.size) / sr
axes[0].plot(t[:600], audio[:600], color="0.15", linewidth=0.8)
axes[0].set_title("原始波形")
axes[1].plot(t[:600], decoded[:600], color="0.15", linewidth=0.8)
axes[1].set_title("μ-law 解码波形")
axes[1].set_xlabel("时间（秒）")
finish_figure(fig, OUTPUT_FIGURES / "08_2_mulaw_waveform.png")
plt.show()

display(Audio(str(OUTPUT_AUDIO / "harmonic_mulaw_decoded.wav")))


In [ ]:
model = WaveNet(
    quantization_channels=256,
    residual_channels=16,
    dilation_channels=16,
    skip_channels=32,
    dilations=(1, 2, 4, 8),
).eval()

x = torch.as_tensor(tokens[:512], dtype=torch.long).unsqueeze(0)
with torch.no_grad():
    logits = model(x)

print("input tokens:", tuple(x.shape))
print("WaveNet logits:", tuple(logits.shape))
assert logits.shape == (1, 256, 512)


In [ ]:
changed_future = x.clone()
changed_future[:, 300:] = (changed_future[:, 300:] + 17) % 256
with torch.no_grad():
    base_logits = model(x)
    changed_logits = model(changed_future)

max_diff_before_change = (base_logits[:, :, :300] - changed_logits[:, :, :300]).abs().max().item()
print("max logit diff before changed future:", max_diff_before_change)
assert max_diff_before_change < 1e-6


In [ ]:
import librosa

n_fft, hop_length = 1024, 256
specs = [
    np.abs(librosa.stft(audio, n_fft=n_fft, hop_length=hop_length)),
    np.abs(librosa.stft(decoded, n_fft=n_fft, hop_length=hop_length)),
]
ref = specs[0].max()
dbs = [librosa.amplitude_to_db(spec, ref=ref) for spec in specs]

fig, axes = plt.subplots(2, 1, figsize=(9, 5.6), sharex=True, sharey=True, layout="constrained")
extent = [0, dbs[0].shape[1] * hop_length / sr, 0, sr / 2]
titles = ["原始谐波信号", "μ-law 解码谐波信号"]
for ax, db, panel_title in zip(axes, dbs, titles):
    image = ax.imshow(db, origin="lower", aspect="auto", extent=extent, cmap="gray_r")
    ax.set_title(panel_title)
    ax.set_ylabel("频率（Hz）")
axes[1].set_xlabel("时间（秒）")
fig.colorbar(image, ax=axes, fraction=0.046, pad=0.04)
finish_figure(fig, OUTPUT_FIGURES / "08_2_mulaw_spectrogram.png", tight=False)
plt.show()

print("08_2 generated figures:")
for path in sorted(OUTPUT_FIGURES.glob("08_2_*.png")):
    print("-", path.name)
